# Agreement Audit (Colab)

Checks every agreement linked in a Google Sheet for **customer (society) name**, **stamp** and **signature**,
using cheap deterministic checks first and a **Llama 3.2 90B Vision** model only for what's still unknown.

Sheet layout: `B` = document link → the notebook fills `C` Society Name, `D` Is Stamped, `E` Is Signed, `F` Notes.

**Before running:** add your NVIDIA key in Colab → 🔑 *Secrets* as `NVIDIA_API_KEY` (never paste keys into code).

In [ ]:
# 1. Install + get the code
!pip install -q pymupdf gspread google-api-python-client google-auth pillow requests
!git clone -q https://github.com/prabhat620534-png/agreement-audit-vision.git
import sys; sys.path.insert(0, "agreement-audit-vision/src")

In [ ]:
# 2. Settings - key comes from Colab Secrets, not from the notebook
import os
from google.colab import userdata
os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")
os.environ["VENDOR_NAME"] = "Acme Services Pvt. Ltd."      # the company issuing the agreements
os.environ["VENDOR_PATTERN"] = r"acme\s*services"         # never returned as the customer name

from agreement_audit.config import Settings
settings = Settings()
print("Model:", settings.model)

In [ ]:
# 3. (Optional) try it offline on the bundled fictional samples - no API calls
!cd agreement-audit-vision/src && python -m agreement_audit.cli ../samples --no-vision --out /content/sample_report.csv
import pandas as pd; pd.read_csv("/content/sample_report.csv")

In [ ]:
# 4. Connect to your Google Sheet
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/<your-sheet-id>/edit"
ws = gc.open_by_url(SPREADSHEET_URL).worksheet("Sheet1")
print("Connected:", ws.title, ws.row_count, "rows")

In [ ]:
# 5. Run (resumable - re-run to continue or retry errors). Start with test_limit=5.
from agreement_audit.sheets import run_sheet
run_sheet(ws, creds, test_limit=5, s=settings)